In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import json
import numpy as np
import pandas as pd

In [2]:
model_id = "Zual/MPropositioneur-V2-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
inputs = tokenizer("Voici la proposition à analyser.", return_tensors="pt").to(device)

with torch.no_grad():
    # output_hidden_states=True est impératif pour récupérer les embeddings
    outputs = model(**inputs, output_hidden_states=True)

# Récupération de l'avant-dernière couche
penultimate_layer = outputs.hidden_states[-2]

# Extraction du vecteur du dernier token (pour la première phrase du batch)
# Dimension résultante : [taille_du_vecteur_caché]
sentence_embedding = penultimate_layer[0, -1, :]

torch.Size([2560])